### Préparation : Importation des données et des librairies

In [10]:
### Importation des librairies ###


import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path
from fonction import *
import seaborn as sns
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

In [11]:
### Importation de la base ###


# Importation
gdf = gpd.read_file('data/processed/jointure_meteo_swimonthly_full.gpkg')

# Affichage
display(gdf)

,NUMERO,LAMBX,LAMBY,DATE,SWI_UNIF_MENS,PRENEI,PRELIQ,T,FF,Q,...,HTEURNEIGE,HTEURNEIGE6,HTEURNEIGEX,SNOW_FRAC,ECOULEMENT,WG_RACINE,WGI_RACINE,TINF_H,TSUP_H,geometry
0,2,641374,7106309,1960-01-01,0.863,4.9,51.5,4.829032,6.822581,4.584839,...,0.001710,0.001839,0.024,0.096774,4.0,0.287032,0.002484,-7.2,11.6,POINT (588001.454 2673000.704)
1,7119,635809,6442801,1960-01-01,1.081,20.2,96.4,1.990323,2.061290,3.926677,...,0.030935,0.031323,0.174,0.177419,38.0,0.298742,0.012194,-13.1,11.7,POINT (587999.304 2008998.388)
2,7118,627817,6442868,1960-01-01,1.100,22.7,119.8,2.783871,1.996774,4.188290,...,0.032935,0.033581,0.180,0.203226,50.2,0.264548,0.012129,-12.8,12.2,POINT (579999.298 2008998.399)
3,7117,619825,6442935,1960-01-01,1.095,21.8,115.4,3.277419,2.083871,4.315355,...,0.030839,0.031226,0.170,0.180645,45.7,0.284097,0.010065,-12.2,13.0,POINT (571999.291 2008998.458)
4,7116,611833,6443002,1960-01-01,1.086,23.8,108.4,3.490323,2.174194,4.422677,...,0.032871,0.033323,0.179,0.183871,41.7,0.293774,0.008742,-11.5,13.3,POINT (563999.284 2008998.565)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7005175,3831,805955,6713138,2024-12-01,0.927,0.9,68.4,4.016129,2.809677,4.880258,...,0.000000,0.000000,0.000,0.000000,0.9,0.344613,0.000097,-3.9,12.8,POINT (756000.273 2280998.1)
7005176,3832,813948,6713070,2024-12-01,0.908,1.5,71.1,2.861290,3.348387,4.495484,...,0.000000,0.000000,0.001,0.000000,1.8,0.324968,0.000613,-5.0,11.4,POINT (763999.644 2280998.186)
7005177,3833,821942,6713002,2024-12-01,0.923,3.2,71.2,2.835484,3.409677,4.498032,...,0.000000,0.000000,0.001,0.000000,3.9,0.321000,0.000516,-4.8,11.5,POINT (772000.016 2280998.33)
7005178,3827,773980,6713410,2024-12-01,0.973,0.0,64.9,4.154839,2.825806,4.791226,...,0.000000,0.000000,0.000,0.000000,0.0,0.375032,0.000774,-2.5,13.3,POINT (723999.802 2280998.23)


In [12]:
### Affichage de la liste des variables ###


list(gdf.columns)

['NUMERO',
 'LAMBX',
 'LAMBY',
 'DATE',
 'SWI_UNIF_MENS',
 'PRENEI',
 'PRELIQ',
 'T',
 'FF',
 'Q',
 'DLI',
 'SSI',
 'HU',
 'EVAP',
 'ETP',
 'PE',
 'SWI',
 'SSWI_10J',
 'DRAINC',
 'RUNC',
 'RESR_NEIGE',
 'RESR_NEIGE6',
 'HTEURNEIGE',
 'HTEURNEIGE6',
 'HTEURNEIGEX',
 'SNOW_FRAC',
 'ECOULEMENT',
 'WG_RACINE',
 'WGI_RACINE',
 'TINF_H',
 'TSUP_H',
 'geometry']

In [13]:
### Isolement de la variable de SWI uniforme mensuel (y) et des variables météorologiques ###


# Variable de SWI uniforme mensuel
y = gdf["SWI_UNIF_MENS"]

# Variables météorologiques
meteo_vars = [
    "T", "Q", "FF",
    "PRENEI", "PRELIQ",
    "SSI", "DLI",
    "HTEURNEIGE", "SNOW_FRAC",
    "EVAP", "ETP",
    "PE", "DRAINC", "RUNC",
    "RESR_NEIGE", "RESR_NEIGE6",
    "HTEURNEIGE", "HTEURNEIGE6", "HTEURNEIGEX",
    "SNOW_FRAC", "ECOULEMENT",
    "WG_RACINE", "WGI_RACINE",
    "TINF_H", "TSUP_H"
]

### Enrichissement de la base

In [14]:
### Ajout d'une colonne de SWI des deux périodes précédentes (lag 1 et lag 2) ###


# Tri de la base par point de la grille puis par date
gdf = gdf.sort_values(by=["NUMERO", "DATE"])

# Ajout de la colonne SWI_UNIF_MENS_LAG1
gdf["SWI_UNIF_MENS_LAG1"] = gdf.groupby("NUMERO")["SWI_UNIF_MENS"].shift(1)

# Ajout de la colonne SWI_UNIF_MENS_LAG2
gdf["SWI_UNIF_MENS_LAG2"] = gdf.groupby("NUMERO")["SWI_UNIF_MENS"].shift(2)

# Affichage de la base
display(gdf)

,NUMERO,LAMBX,LAMBY,DATE,SWI_UNIF_MENS,PRENEI,PRELIQ,T,FF,Q,...,HTEURNEIGEX,SNOW_FRAC,ECOULEMENT,WG_RACINE,WGI_RACINE,TINF_H,TSUP_H,geometry,SWI_UNIF_MENS_LAG1,SWI_UNIF_MENS_LAG2
0,2,641374,7106309,1960-01-01,0.863,4.9,51.5,4.829032,6.822581,4.584839,...,0.024,0.096774,4.0,0.287032,0.002484,-7.2,11.6,POINT (588001.454 2673000.704),NaN,NaN
15717,2,641374,7106309,1960-02-01,0.876,0.5,35.9,5.113793,5.631034,4.415690,...,0.001,0.000000,0.5,0.276517,0.000379,-0.7,17.9,POINT (588001.454 2673000.704),0.863,NaN
23945,2,641374,7106309,1960-03-01,0.856,0.0,43.6,6.464516,5.490323,4.924000,...,0.000,0.000000,0.0,0.268097,0.000000,-0.8,14.6,POINT (588001.454 2673000.704),0.876,0.863
33679,2,641374,7106309,1960-04-01,0.757,0.0,6.8,9.030000,6.200000,5.476100,...,0.000,0.000000,0.0,0.254867,0.000000,3.5,20.4,POINT (588001.454 2673000.704),0.856,0.876
41906,2,641374,7106309,1960-05-01,0.673,0.0,93.1,12.261290,3.712903,7.106710,...,0.000,0.000000,0.0,0.253484,0.000000,1.7,23.5,POINT (588001.454 2673000.704),0.757,0.856
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6964764,9892,1215772,6046242,2024-08-01,-0.019,0.0,23.9,26.196774,2.651613,14.750645,...,0.000,0.000000,0.0,0.190419,0.000000,17.1,37.7,POINT (1171999.296 1617001.255),-0.019,0.057
6971499,9892,1215772,6046242,2024-09-01,0.007,0.0,64.6,20.936667,4.023333,11.472100,...,0.000,0.000000,0.0,0.196900,0.000000,8.2,32.9,POINT (1171999.296 1617001.255),-0.019,-0.019
6981220,9892,1215772,6046242,2024-10-01,0.170,0.0,100.2,19.009677,3.525806,11.309484,...,0.000,0.000000,0.0,0.224065,0.000000,7.0,30.4,POINT (1171999.296 1617001.255),0.007,-0.019
6988708,9892,1215772,6046242,2024-11-01,0.126,0.0,16.4,14.446667,3.973333,7.915033,...,0.000,0.000000,0.0,0.219767,0.000000,0.1,24.1,POINT (1171999.296 1617001.255),0.170,0.007


In [15]:
### Ajout de colonnes de variables climatiques des périodes précédentes (lag) ###


# WGRACINE et WGI_RACINE lag 1
gdf["WG_RACINE_LAG1"] = gdf.groupby("NUMERO")["WG_RACINE"].shift(1)
gdf["WGI_RACINE_LAG1"] = gdf.groupby("NUMERO")["WGI_RACINE"].shift(1)

# DRAINC, RUNC et ECOULEMENT lag 1
gdf["DRAINC_LAG1"] = gdf.groupby("NUMERO")["DRAINC"].shift(1)
gdf["RUNC_LAG1"] = gdf.groupby("NUMERO")["RUNC"].shift(1)
gdf["ECOULEMENT_LAG1"] = gdf.groupby("NUMERO")["ECOULEMENT"].shift(1)

# RESR_NEIGE, HTEURNEIGE et SNOW_FRAC lag 1
gdf["RESR_NEIGE_LAG1"] = gdf.groupby("NUMERO")["RESR_NEIGE"].shift(1)
gdf["HTEURNEIGE_LAG1"] = gdf.groupby("NUMERO")["HTEURNEIGE"].shift(1)
gdf["SNOW_FRAC_LAG1"] = gdf.groupby("NUMERO")["SNOW_FRAC"].shift(1)

# Affichage de la base
display(gdf)

,NUMERO,LAMBX,LAMBY,DATE,SWI_UNIF_MENS,PRENEI,PRELIQ,T,FF,Q,...,SWI_UNIF_MENS_LAG1,SWI_UNIF_MENS_LAG2,WG_RACINE_LAG1,WGI_RACINE_LAG1,DRAINC_LAG1,RUNC_LAG1,ECOULEMENT_LAG1,RESR_NEIGE_LAG1,HTEURNEIGE_LAG1,SNOW_FRAC_LAG1
0,2,641374,7106309,1960-01-01,0.863,4.9,51.5,4.829032,6.822581,4.584839,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15717,2,641374,7106309,1960-02-01,0.876,0.5,35.9,5.113793,5.631034,4.415690,...,0.863,NaN,0.287032,0.002484,27.4,5.7,4.0,0.287097,0.00171,0.096774
23945,2,641374,7106309,1960-03-01,0.856,0.0,43.6,6.464516,5.490323,4.924000,...,0.876,0.863,0.276517,0.000379,11.2,2.8,0.5,0.000000,0.00000,0.000000
33679,2,641374,7106309,1960-04-01,0.757,0.0,6.8,9.030000,6.200000,5.476100,...,0.856,0.876,0.268097,0.000000,9.2,3.4,0.0,0.000000,0.00000,0.000000
41906,2,641374,7106309,1960-05-01,0.673,0.0,93.1,12.261290,3.712903,7.106710,...,0.757,0.856,0.254867,0.000000,4.3,0.2,0.0,0.000000,0.00000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6964764,9892,1215772,6046242,2024-08-01,-0.019,0.0,23.9,26.196774,2.651613,14.750645,...,-0.019,0.057,0.194871,0.000000,3.1,0.1,0.0,0.000000,0.00000,0.000000
6971499,9892,1215772,6046242,2024-09-01,0.007,0.0,64.6,20.936667,4.023333,11.472100,...,-0.019,-0.019,0.190419,0.000000,3.1,0.0,0.0,0.000000,0.00000,0.000000
6981220,9892,1215772,6046242,2024-10-01,0.170,0.0,100.2,19.009677,3.525806,11.309484,...,0.007,-0.019,0.196900,0.000000,3.0,0.9,0.0,0.000000,0.00000,0.000000
6988708,9892,1215772,6046242,2024-11-01,0.126,0.0,16.4,14.446667,3.973333,7.915033,...,0.170,0.007,0.224065,0.000000,3.1,4.5,0.0,0.000000,0.00000,0.000000


In [16]:
### Ajout de variables climatiques de cumul de précipitations et de différence de neige ###


# Cumul des précipitations PRENEI, PRELIQ et PE sur les 3 derniers mois
gdf["PRENEI_CUM3"] = gdf.groupby("NUMERO")["PRENEI"].rolling(window=3).sum().reset_index(level = 0, drop = True)
gdf["PRELIQ_CUM3"] = gdf.groupby("NUMERO")["PRELIQ"].rolling(window=3).sum().reset_index(level = 0, drop = True)
gdf["PE_CUM3"] = gdf.groupby("NUMERO")["PE"].rolling(window=3).sum().reset_index(level = 0, drop = True)

# Différence de neige RESR_NEIGE, HTEURNEIGE et SNOWFRAC entre le mois courant et le mois précédent
gdf["RESR_NEIGE_DIFF1"] = gdf["RESR_NEIGE"] - gdf["RESR_NEIGE_LAG1"]
gdf["HTEURNEIGE_DIFF1"] = gdf["HTEURNEIGE"] - gdf["HTEURNEIGE_LAG1"]
gdf["SNOW_FRAC_DIFF1"] = gdf["SNOW_FRAC"] - gdf["SNOW_FRAC_LAG1"]

# Affichage de la base
display(gdf)

,NUMERO,LAMBX,LAMBY,DATE,SWI_UNIF_MENS,PRENEI,PRELIQ,T,FF,Q,...,ECOULEMENT_LAG1,RESR_NEIGE_LAG1,HTEURNEIGE_LAG1,SNOW_FRAC_LAG1,PRENEI_CUM3,PRELIQ_CUM3,PE_CUM3,RESR_NEIGE_DIFF1,HTEURNEIGE_DIFF1,SNOW_FRAC_DIFF1
0,2,641374,7106309,1960-01-01,0.863,4.9,51.5,4.829032,6.822581,4.584839,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15717,2,641374,7106309,1960-02-01,0.876,0.5,35.9,5.113793,5.631034,4.415690,...,4.0,0.287097,0.00171,0.096774,NaN,NaN,NaN,-0.287097,-0.00171,-0.096774
23945,2,641374,7106309,1960-03-01,0.856,0.0,43.6,6.464516,5.490323,4.924000,...,0.5,0.000000,0.00000,0.000000,5.4,131.0,35.9,0.000000,0.00000,0.000000
33679,2,641374,7106309,1960-04-01,0.757,0.0,6.8,9.030000,6.200000,5.476100,...,0.0,0.000000,0.00000,0.000000,0.5,86.3,-13.2,0.000000,0.00000,0.000000
41906,2,641374,7106309,1960-05-01,0.673,0.0,93.1,12.261290,3.712903,7.106710,...,0.0,0.000000,0.00000,0.000000,0.0,143.5,19.7,0.000000,0.00000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6964764,9892,1215772,6046242,2024-08-01,-0.019,0.0,23.9,26.196774,2.651613,14.750645,...,0.0,0.000000,0.00000,0.000000,0.0,48.9,-33.0,0.000000,0.00000,0.000000
6971499,9892,1215772,6046242,2024-09-01,0.007,0.0,64.6,20.936667,4.023333,11.472100,...,0.0,0.000000,0.00000,0.000000,0.0,100.8,-6.7,0.000000,0.00000,0.000000
6981220,9892,1215772,6046242,2024-10-01,0.170,0.0,100.2,19.009677,3.525806,11.309484,...,0.0,0.000000,0.00000,0.000000,0.0,188.7,55.6,0.000000,0.00000,0.000000
6988708,9892,1215772,6046242,2024-11-01,0.126,0.0,16.4,14.446667,3.973333,7.915033,...,0.0,0.000000,0.00000,0.000000,0.0,181.2,38.8,0.000000,0.00000,0.000000


In [17]:
### Liste des variables climatiques ajoutées ###


meteo_vars_added = [
    "SWI_UNIF_MENS_LAG1", "SWI_UNIF_MENS_LAG2",
    "WG_RACINE_LAG1", "WGI_RACINE_LAG1",
    "DRAINC_LAG1", "RUNC_LAG1", "ECOULEMENT_LAG1",
    "RESR_NEIGE_LAG1", "HTEURNEIGE_LAG1", "SNOW_FRAC_LAG1",
    "PRENEI_CUM3", "PRELIQ_CUM3", "PE_CUM3",
    "RESR_NEIGE_DIFF1", "HTEURNEIGE_DIFF1", "SNOW_FRAC_DIFF1"
]

In [18]:
### Ajout du cosinus et du sinus à partir du mois ###


# Copie du dataframe
gdf = gdf.copy()

# Extraction du mois
gdf["month"] = gdf["DATE"].dt.month

# Calcul du sinus et du cosinus à partir du mois pour capturer la saisonnalité
gdf["sin_month"] = np.sin(2 * np.pi * gdf["month"] / 12)
gdf["cos_month"] = np.cos(2 * np.pi * gdf["month"] / 12)

# Coordonnées x et y des points
gdf["x"] = gdf.geometry.x
gdf["y"] = gdf.geometry.y

In [20]:
### Dataframes des variables explicatives X et de la variable cible Y ###


# Liste des variables explicatives
X_cols = (
    meteo_vars +
    meteo_vars_added +
    ["sin_month", "cos_month", "x", "y"]
)

# Dataframes X et y
X = gdf[X_cols]
y = gdf["SWI_UNIF_MENS"]

MemoryError: Unable to allocate 2.14 GiB for an array with shape (41, 7005180) and data type float64

In [10]:
X

,T,Q,FF,PRENEI,PRELIQ,SSI,DLI,HTEURNEIGE,SNOW_FRAC,ECOULEMENT,WG_RACINE,WGI_RACINE,TINF_H,TSUP_H,sin_month,cos_month,x,y
0,4.829032,4.584839,6.822581,4.9,51.5,5668.6,85952.0,0.001710,0.096774,4.0,0.287032,0.002484,-7.2,11.6,5.000000e-01,0.866025,5.880015e+05,2.673001e+06
1,1.990323,3.926677,2.061290,20.2,96.4,14448.3,73603.7,0.030935,0.177419,38.0,0.298742,0.012194,-13.1,11.7,5.000000e-01,0.866025,5.879993e+05,2.008998e+06
2,2.783871,4.188290,1.996774,22.7,119.8,15546.2,71915.4,0.032935,0.203226,50.2,0.264548,0.012129,-12.8,12.2,5.000000e-01,0.866025,5.799993e+05,2.008998e+06
3,3.277419,4.315355,2.083871,21.8,115.4,15365.5,72581.4,0.030839,0.180645,45.7,0.284097,0.010065,-12.2,13.0,5.000000e-01,0.866025,5.719993e+05,2.008998e+06
4,3.490323,4.422677,2.174194,23.8,108.4,15037.1,73627.8,0.032871,0.183871,41.7,0.293774,0.008742,-11.5,13.3,5.000000e-01,0.866025,5.639993e+05,2.008999e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7005175,4.016129,4.880258,2.809677,0.9,68.4,10204.5,84831.5,0.000000,0.000000,0.9,0.344613,0.000097,-3.9,12.8,-2.449294e-16,1.000000,7.560003e+05,2.280998e+06
7005176,2.861290,4.495484,3.348387,1.5,71.1,9272.6,84829.3,0.000000,0.000000,1.8,0.324968,0.000613,-5.0,11.4,-2.449294e-16,1.000000,7.639996e+05,2.280998e+06
7005177,2.835484,4.498032,3.409677,3.2,71.2,9420.0,83560.9,0.000000,0.000000,3.9,0.321000,0.000516,-4.8,11.5,-2.449294e-16,1.000000,7.720000e+05,2.280998e+06
7005178,4.154839,4.791226,2.825806,0.0,64.9,8200.5,81367.8,0.000000,0.000000,0.0,0.375032,0.000774,-2.5,13.3,-2.449294e-16,1.000000,7.239998e+05,2.280998e+06


In [11]:
train = gdf[gdf["DATE"] < "2018-01-01"]
test  = gdf[gdf["DATE"] >= "2018-01-01"]

X_train = train[X_cols]
y_train = train["SWI_UNIF_MENS"]

X_test = test[X_cols]
y_test = test["SWI_UNIF_MENS"]

In [ ]:
model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("reg", Ridge(alpha=1.0))
    ]
)

In [28]:
from sklearn import metrics

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# Print the complete summary of the model's performance
print("Model coefficients:", model.named_steps['reg'].coef_)
print("Model intercept:", model.named_steps['reg'].intercept_)
print("R² (train):", model.score(X_train, y_train))
print("R² (test):", model.score(X_test, y_test))
print("RMSE (train):", np.sqrt(metrics.mean_squared_error(y_train, model.predict(X_train))))
print("RMSE (test):", np.sqrt(metrics.mean_squared_error(y_test, y_pred)))

Model coefficients: [-0.20427573  0.11811369 -0.02023296  0.01094149  0.07143714 -0.00892247
 -0.02959015 -0.00955322 -0.02065648  0.02161763  0.094615    0.01332031
  0.0254982  -0.04263763  0.13538385 -0.05074013 -0.0071835   0.00355194]
Model intercept: 0.61735012628832
R² (train): 0.7942477544506177
R² (test): 0.8249452658441506
RMSE (train): 0.14758000081949657
RMSE (test): 0.14276823744045217


In [26]:
y_test

6250776    1.083
6250777    1.083
6250778    1.090
6250779    1.080
6250780    1.174
           ...  
7005175    0.927
7005176    0.908
7005177    0.923
7005178    0.973
7005179    0.177
Name: SWI_UNIF_MENS, Length: 754404, dtype: float64

In [24]:
y_pred

array([1.07484535, 1.07106793, 1.08914599, ..., 0.819295  , 0.90421076,
       0.40410732], shape=(754404,))